# Phase 10 — Ensemble DistilBERT(train+valid) + SVM

**Στρατηγική:**
- Βάρη ensemble: βρίσκονται με τα **παλιά** bert_valid probs (από phase4)
- Test predictions: χρησιμοποιούν τα **νέα** bert_fulldata probs (από phase9)


In [1]:
import numpy as np
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import f1_score
import warnings
warnings.filterwarnings('ignore')

In [2]:
train = pd.read_csv('train.csv')
valid = pd.read_csv('valid.csv')
test  = pd.read_csv('test.csv')

# Παλιά valid probs (phase4) — εύρεση βέλτιστων βαρών
bert_valid_hazard_probs  = np.load('npy/bert_valid_hazard_probs.npy')
bert_valid_product_probs = np.load('npy/bert_valid_product_probs.npy')

# Νέα test probs (phase9, train+valid) — για τελικές προβλέψεις
bert_test_hazard_probs  = np.load('npy/bert_fulldata_test_hazard_probs.npy')
bert_test_product_probs = np.load('npy/bert_fulldata_test_product_probs.npy')

y_valid_hazard  = valid['hazard-category']
y_valid_product = valid['product-category']

print(f'Train: {len(train)} | Valid: {len(valid)} | Test: {len(test)}')
print(f'bert_valid_hazard_probs:  {bert_valid_hazard_probs.shape}')
print(f'bert_valid_product_probs: {bert_valid_product_probs.shape}')
print(f'bert_test_hazard_probs:   {bert_test_hazard_probs.shape}')
print(f'bert_test_product_probs:  {bert_test_product_probs.shape}')

Train: 5082 | Valid: 565 | Test: 997
bert_valid_hazard_probs:  (565, 10)
bert_valid_product_probs: (565, 22)
bert_test_hazard_probs:   (997, 10)
bert_test_product_probs:  (997, 22)


In [3]:
def preprocess_text(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def official_st1_score(y_true_hazard, y_pred_hazard,
                       y_true_product, y_pred_product, verbose=True):
    f1_hazard = f1_score(y_true_hazard, y_pred_hazard, average='macro', zero_division=0)
    y_true_hazard  = np.array(y_true_hazard)
    y_pred_hazard  = np.array(y_pred_hazard)
    y_true_product = np.array(y_true_product)
    y_pred_product = np.array(y_pred_product)
    mask = (y_true_hazard == y_pred_hazard)
    f1_product = f1_score(
        y_true_product[mask], y_pred_product[mask],
        average='macro', zero_division=0
    ) if mask.sum() > 0 else 0.0
    score = (f1_hazard + f1_product) / 2
    if verbose:
        print(f'macro-F1 Hazard:                    {f1_hazard:.4f}')
        print(f'Σωστά hazard predictions:           {mask.sum()}/{len(mask)} ({100*mask.mean():.1f}%)')
        print(f'macro-F1 Product (given correct h): {f1_product:.4f}')
        print(f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
        print(f'OFFICIAL ST1 SCORE:                 {score:.4f}')
    return score

def probability_ensemble(svm_scores, bert_probs, svm_classes, w_svm, w_bert):
    scaler   = MinMaxScaler()
    svm_norm = scaler.fit_transform(svm_scores)
    combined = w_svm * svm_norm + w_bert * bert_probs
    return svm_classes[combined.argmax(axis=1)]

In [4]:
# SVM — εκπαίδευση σε train ΜΟΝΟ (όπως στο phase4)
# ώστε να μπορούμε να αξιολογήσουμε στο valid
train['combined'] = train['title'].fillna('') + ' ' + train['text'].fillna('').str[:550]
valid['combined'] = valid['title'].fillna('') + ' ' + valid['text'].fillna('').str[:550]
test['combined']  = test['title'].fillna('')  + ' ' + test['text'].fillna('').str[:550]

train['combined'] = train['combined'].apply(preprocess_text)
valid['combined'] = valid['combined'].apply(preprocess_text)
test['combined']  = test['combined'].apply(preprocess_text)

tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    stop_words='english'
)
X_train = tfidf.fit_transform(train['combined'])
X_valid = tfidf.transform(valid['combined'])
X_test  = tfidf.transform(test['combined'])

clf_svm_hazard = LinearSVC(C=0.5, class_weight='balanced', max_iter=2000, random_state=42)
clf_svm_hazard.fit(X_train, train['hazard-category'])

clf_svm_product = LinearSVC(C=0.5, class_weight='balanced', max_iter=2000, random_state=42)
clf_svm_product.fit(X_train, train['product-category'])

print('SVM trained')
print(f'TF-IDF features: {X_train.shape[1]}')

SVM trained
TF-IDF features: 50000


In [5]:
# Εύρεση βέλτιστων βαρών χρησιμοποιώντας παλιά valid probs (phase4)
svm_valid_hazard_scores  = clf_svm_hazard.decision_function(X_valid)
svm_valid_product_scores = clf_svm_product.decision_function(X_valid)

weight_combinations = [(0.9,0.1),(0.7,0.3),(0.5,0.5),(0.3,0.7),(0.1,0.9)]

print('=== ΕΎΡΕΣΗ ΒΈΛΤΙΣΤΩΝ ΒΑΡΏΝ (με παλιά valid probs) ===')
print('(Βάρη: SVM, BERT)\n')

best_score   = 0
best_weights = None

for w_svm, w_bert in weight_combinations:
    ens_hazard = probability_ensemble(
        svm_valid_hazard_scores, bert_valid_hazard_probs,
        clf_svm_hazard.classes_, w_svm, w_bert
    )
    ens_product = probability_ensemble(
        svm_valid_product_scores, bert_valid_product_probs,
        clf_svm_product.classes_, w_svm, w_bert
    )
    score = official_st1_score(
        y_valid_hazard, ens_hazard,
        y_valid_product, ens_product,
        verbose=False
    )
    print(f'SVM={w_svm}, BERT={w_bert} → Score: {score:.4f}')
    if score > best_score:
        best_score   = score
        best_weights = (w_svm, w_bert)

print(f'\nΒέλτιστα βάρη: SVM={best_weights[0]}, BERT={best_weights[1]}')
print(f'Validation score με παλιά BERT probs: {best_score:.4f}')

=== ΕΎΡΕΣΗ ΒΈΛΤΙΣΤΩΝ ΒΑΡΏΝ (με παλιά valid probs) ===
(Βάρη: SVM, BERT)

SVM=0.9, BERT=0.1 → Score: 0.6655
SVM=0.7, BERT=0.3 → Score: 0.7402
SVM=0.5, BERT=0.5 → Score: 0.7876
SVM=0.3, BERT=0.7 → Score: 0.8028
SVM=0.1, BERT=0.9 → Score: 0.7998

Βέλτιστα βάρη: SVM=0.3, BERT=0.7
Validation score με παλιά BERT probs: 0.8028


In [6]:
# Τελικές test predictions με νέα BERT probs (train+valid, phase9)
w_svm_best, w_bert_best = best_weights

svm_test_hazard_scores  = clf_svm_hazard.decision_function(X_test)
svm_test_product_scores = clf_svm_product.decision_function(X_test)

ensemble_test_hazard = probability_ensemble(
    svm_test_hazard_scores, bert_test_hazard_probs,
    clf_svm_hazard.classes_, w_svm_best, w_bert_best
)
ensemble_test_product = probability_ensemble(
    svm_test_product_scores, bert_test_product_probs,
    clf_svm_product.classes_, w_svm_best, w_bert_best
)

submission = pd.DataFrame({
    'id': test['id'],
    'hazard-category': ensemble_test_hazard,
    'product-category': ensemble_test_product
})
submission.to_csv('submission_ensemble_fulldata.csv', index=False)

print(f'Αποθηκεύτηκε: submission_ensemble_fulldata.csv')
print(f'Βάρη: SVM={w_svm_best}, BERT={w_bert_best}')
print(f'Predictions: {len(submission)}')
print('\n--- Πρώτες 5 ---')
print(submission.head())
print('\n=== ΣΥΓΚΡΙΣΗ ===')
print(f'Previous best Kaggle:       0.755  (DistilBERT train-only + SVM)')
print(f'DistilBERT train+valid solo: 0.76064')

Αποθηκεύτηκε: submission_ensemble_fulldata.csv
Βάρη: SVM=0.3, BERT=0.7
Predictions: 997

--- Πρώτες 5 ---
   id hazard-category              product-category
0   0      biological  meat, egg and dairy products
1   1      biological  meat, egg and dairy products
2   2      biological    prepared dishes and snacks
3   3      biological  meat, egg and dairy products
4   4  foreign bodies             ices and desserts

=== ΣΥΓΚΡΙΣΗ ===
Previous best Kaggle:       0.755  (DistilBERT train-only + SVM)
DistilBERT train+valid solo: 0.76064
